In [ ]:
# Import packages and initialize Earth Engine

import ee
import geemap
import pandas as pd
import numpy as np
import os
import seaborn as sns

geemap.ee_initialize()

### Initialize variables: Mask, extraction points and time frame 

In [37]:
date_start = ee.Date('2023-11-09') # Latest of all stations: Disko
date_end = ee.Date('2024-01-01') # Latest of all stations: Disko

greenlandmask = ee.Image('OSU/GIMP/2000_ICE_OCEAN_MASK').select('ocean_mask').eq(0)
greenland = ee.Geometry.Polygon(
[[[-36.29516924635421, 83.70737243835941],
[-51.85180987135421, 82.75597137647488],
[-61.43188799635421, 81.99879137488564],
[-74.08813799635422, 78.10103528196419],
[-70.13305987135422, 75.65372336709613],
[-61.08032549635421, 75.71891096312955],
[-52.20337237135421, 60.9795530382023],
[-43.41430987135421, 58.59235996703347],
[-38.49243487135421, 64.70478286561182],
[-19.771731746354217, 69.72271161037442],
[-15.728762996354217, 76.0828635948066],
[-15.904544246354217, 79.45091003031243],
[-10.015872371354217, 81.62328742628017],
[-26.627200496354217, 83.43179828852398],
[-31.636966121354217, 83.7553561747887]]])

awsPoints = ee.FeatureCollection([

# Kobbefjord
ee.Feature(ee.Geometry.Point([-51.37199020385742, 64.12248229980469]), {"id": 'Kobbefjord_M500'}),

# Disko
ee.Feature(ee.Geometry.Point([-53.479400634765625, 69.27300262451172]), {"id": 'Disko_T1'}),
ee.Feature(ee.Geometry.Point([-53.43281936645508, 69.28909301757812]), {"id": 'Disko_T2'}),
ee.Feature(ee.Geometry.Point([-53.45709991455078, 69.2767105102539]), {"id": 'Disko_T3'}),
ee.Feature(ee.Geometry.Point([-53.49897003173828, 69.25126647949219]), {"id": 'Disko_T4'}),

# Zackenberg
ee.Feature(ee.Geometry.Point([-20.563194274902344, 74.46549224853516]), {"id": 'Zackenberg_M2'}),
ee.Feature(ee.Geometry.Point([-20.459354400634766, 74.50310516357422]), {"id": 'Zackenberg_M3'}),
ee.Feature(ee.Geometry.Point([-20.552143096923828, 74.47307586669922]), {"id": 'Zackenberg_M4_30min'})
])


### Create test images 

In [ ]:
def lst(image):
    lst_day = image.select('LST_Day_1km').multiply(0.02).subtract(273.15).rename('LST_Day_1km_C')
    lst_night = image.select('LST_Night_1km').multiply(0.02).subtract(273.15).rename('LST_Night_1km_C')
    return image.addBands(lst_day).addBands(lst_night)

test_image = ee.ImageCollection('MODIS/061/MOD11A1').filterDate(date_start, date_end).first()
test_image2 = ee.Image('MODIS/061/MOD11A1/2012_06_05')
test_image = lst(test_image)
test_image2 = lst(test_image2)

img_col = (
    ee.ImageCollection('MODIS/061/MOD11A1')
    .filterDate(date_start, date_end)
    .filterBounds(greenland)
)

# img_col


### Cloud Masking

In [ ]:
# 2. Load MODIS Terra and Aqua LST data
# 2.1 Quality Control Function
# "ref": "https":#gis.stackexchange.com/a/360887
def bitwiseExtract(input, fromBit, toBit):
    maskSize = ee.Number(1).add(toBit).subtract(fromBit)
    mask = ee.Number(1).leftShift(maskSize).subtract(1)
    return input.rightShift(fromBit).bitwiseAnd(mask)

def maskQualityDaytime(image):
    qa = image.select('QC_Day')

    # Bits 0-"1": Mandatory QA flags
    # "0": LST produced, good quality, not necessary to examine more detailed QA
    # "1": LST produced, other quality, recommend examination of more detailed QA
    # "2": LST not produced due to cloud effects
    # "3": LST not produced primarily due to reasons other than cloud
    # bits01Mask = bitwiseExtract(qa, 0, 1).eq(0)
    bits01Mask = bitwiseExtract(qa, 0, 1).lte(1); 
    # Bits 2-"3": Data quality flag
    # "0": Good data quality
    # "1": Other quality data
    # "2": TBD
    # "3": TBD
    bits23Mask = bitwiseExtract(qa, 2, 3).eq(0)
    # Bits 4-"5": Emissivity error flag
    # "0": Average emissivity error <= 0.01
    # "1": 0.01 < Average emissivity error <= 0.02
    # "2": 0.02 < Average emissivity error <= 0.04
    # "3": Average emissivity error > 0.04
    bits45Mask = bitwiseExtract(qa, 4, 5).eq(0)
    # Bit 6-"7": LST error flag
    # "0": Average LST error <= 1K
    # "1": Average LST error <= 2K
    # "2": Average LST error <= 3K
    # "3": Average LST error > 3K
    bit6Mask = bitwiseExtract(qa, 6, 7).eq(0)

    mask = bits01Mask.And(bits23Mask).And(bits45Mask).And(bit6Mask)

    return image.updateMask(mask)

def maskQualityNighttime(image):
    qa = image.select('QC_Night')

    # Bits 0-"1": Mandatory QA flags
    # "0": LST produced, good quality, not necessary to examine more detailed QA
    # "1": LST produced, other quality, recommend examination of more detailed QA
    # "2": LST not produced due to cloud effects
    # "3": LST not produced primarily due to reasons other than cloud
    # bits01Mask = bitwiseExtract(qa, 0, 1).eq(0)
    bits01Mask = bitwiseExtract(qa, 0, 1).lte(1); 
    # Bits 2-"3": Data quality flag
    # "0": Good data quality
    # "1": Other quality data
    # "2": TBD
    # "3": TBD
    bits23Mask = bitwiseExtract(qa, 2, 3).eq(0)
    # Bits 4-"5": Emissivity error flag
    # "0": Average emissivity error <= 0.01
    # "1": 0.01 < Average emissivity error <= 0.02
    # "2": 0.02 < Average emissivity error <= 0.04
    # "3": Average emissivity error > 0.04
    bits45Mask = bitwiseExtract(qa, 4, 5).eq(0)
    # Bit 6-"7": LST error flag
    # "0": Average LST error <= 1K
    # "1": Average LST error <= 2K
    # "2": Average LST error <= 3K
    # "3": Average LST error > 3K
    bit6Mask = bitwiseExtract(qa, 6, 7).eq(0)

    mask = bits01Mask.And(bits23Mask).And(bits45Mask).And(bit6Mask)

    return image.updateMask(mask)


### Image collection
- band selection
- Temperature conversion
- load modis collections terra + aqua 
- filter Date
- filter greenland 
- apply QA mask
- apply temperature conversion



In [ ]:
# # Functions for band selection and conversion
# def lst_mod_day(image):
#     'Terra Day band selection and conversion'
#     lst_day = image.select('LST_Day_1km').multiply(0.02).subtract(273.15).rename('MOD_LST_Day')
#     qa_day = image.select('QC_Day').rename('MOD_QA_Day')
#     return image.addBands(lst_day).addBands(qa_day)

# def lst_mod_night(image):
#     'Terra Night band selection and conversion'
#     lst_night = image.select('LST_Night_1km').multiply(0.02).subtract(273.15).rename('MOD_LST_Night')
#     qa_night = image.select('QC_Night').rename('MOD_QA_Night')
#     return image.addBands(lst_night).addBands(qa_night)

# def lst_myd_day(image):
#     'Aqua Day band selection and conversion'
#     lst_day = image.select('LST_Day_1km').multiply(0.02).subtract(273.15).rename('MYD_LST_Day')
#     qa_day = image.select('QC_Day').rename('MYD_QA_Day')
#     return image.addBands(lst_day).addBands(qa_day)

# def lst_myd_night(image):
#     'Aqua Night band selection and conversion'
#     lst_night = image.select('LST_Night_1km').multiply(0.02).subtract(273.15).rename('MYD_LST_Night')
#     qa_night = image.select('QC_Night').rename('MYD_QA_Night')
#     return image.addBands(lst_night).addBands(qa_night)

# # Load MODIS Terra and Aqua data, apply quality control and conversion functions
# MOD11A1Daytime = (
#     ee.ImageCollection('MODIS/061/MOD11A1')
#     .select(['LST_Day_1km', 'QC_Day'])
#     .filterDate(date_start, date_end)
#     .filterBounds(greenland)
#     .map(maskQualityDaytime)
#     .map(lst_mod_day)
# )

# MOD11A1Nighttime = (
#     ee.ImageCollection('MODIS/061/MOD11A1')
#     .select(['LST_Night_1km', 'QC_Night'])
#     .filterDate(date_start, date_end)
#     .filterBounds(greenland)
#     .map(maskQualityNighttime)
#     .map(lst_mod_night)
# )

# MYD11A1Daytime = (
#     ee.ImageCollection('MODIS/061/MYD11A1')
#     .select(['LST_Day_1km', 'QC_Day'])
#     .filterDate(date_start, date_end)
#     .filterBounds(greenland)
#     .map(maskQualityDaytime)
#     .map(lst_myd_day)
# )

# MYD11A1Nighttime = (
#     ee.ImageCollection('MODIS/061/MYD11A1')
#     .select(['LST_Night_1km', 'QC_Night'])
#     .filterDate(date_start, date_end)
#     .filterBounds(greenland)
#     .map(maskQualityNighttime)
#     .map(lst_myd_night)
# )


# # # 2.3 Load ERA5 Land data (surface net solar radiation) and convert to daily average
# # ERA5Land = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY') \
# # .select('surface_net_solar_radiation', 'skin_temperature') \
# # .filterDate(startDate, endDate) \
# # .filterBounds(greenland)

# # def func_hkp(image):
# #     return image.updateMask(greenlandmask) \
# # .map(func_hkp)


In [ ]:
# Functions for band selection and conversion
def lst_mod_day(image):
    'Terra Day band selection and conversion'
    lst_day = image.select('LST_Day_1km').multiply(0.02).subtract(273.15).rename('MOD_LST_Day')
    qa_day = image.select('QC_Day').rename('MOD_QA_Day')
    return image.addBands(lst_day).addBands(qa_day)

def lst_mod_night(image):
    'Terra Night band selection and conversion'
    lst_night = image.select('LST_Night_1km').multiply(0.02).subtract(273.15).rename('MOD_LST_Night')
    qa_night = image.select('QC_Night').rename('MOD_QA_Night')
    return image.addBands(lst_night).addBands(qa_night)

def lst_myd_day(image):
    'Aqua Day band selection and conversion'
    lst_day = image.select('LST_Day_1km').multiply(0.02).subtract(273.15).rename('MYD_LST_Day')
    qa_day = image.select('QC_Day').rename('MYD_QA_Day')
    return image.addBands(lst_day).addBands(qa_day)

def lst_myd_night(image):
    'Aqua Night band selection and conversion'
    lst_night = image.select('LST_Night_1km').multiply(0.02).subtract(273.15).rename('MYD_LST_Night')
    qa_night = image.select('QC_Night').rename('MYD_QA_Night')
    return image.addBands(lst_night).addBands(qa_night)

# Load MODIS Terra and Aqua data, apply quality control and conversion functions
MOD11A1Daytime = (
    ee.ImageCollection('MODIS/061/MOD11A1')
    .select(['LST_Day_1km', 'QC_Day'])
    .filterDate(date_start, date_end)
    .filterBounds(greenland)
    .map(maskQualityDaytime)
    .map(lst_mod_day)
)

MOD11A1Nighttime = (
    ee.ImageCollection('MODIS/061/MOD11A1')
    .select(['LST_Night_1km', 'QC_Night'])
    .filterDate(date_start, date_end)
    .filterBounds(greenland)
    .map(maskQualityNighttime)
    .map(lst_mod_night)
)

MYD11A1Daytime = (
    ee.ImageCollection('MODIS/061/MYD11A1')
    .select(['LST_Day_1km', 'QC_Day'])
    .filterDate(date_start, date_end)
    .filterBounds(greenland)
    .map(maskQualityDaytime)
    .map(lst_myd_day)
)

MYD11A1Nighttime = (
    ee.ImageCollection('MODIS/061/MYD11A1')
    .select(['LST_Night_1km', 'QC_Night'])
    .filterDate(date_start, date_end)
    .filterBounds(greenland)
    .map(maskQualityNighttime)
    .map(lst_myd_night)
)

# # Create a stack and select only images that cover aws points from stack
# modis_aws = (
#     MOD11A1Daytime.merge(MOD11A1Nighttime).merge(MYD11A1Daytime).merge(MYD11A1Nighttime)
#     .filterBounds(awsPoints)
# )

# # 2.3 Load ERA5 Land data (surface net solar radiation) and convert to daily average
# ERA5Land = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY') \
# .select('surface_net_solar_radiation', 'skin_temperature') \
# .filterDate(startDate, endDate) \
# .filterBounds(greenland)

# def func_hkp(image):
#     return image.updateMask(greenlandmask) \
# .map(func_hkp)




### Extraction of MODIS LST at AWS locations
- Create buffer of 1000m around aws points
- Create function for extraction of zonal statistics, including meta data and params. 
- Extract 

In [34]:
# ATTEMPT 1 - Zonal Statistics Function with defaults

# Create a buffer function 
def bufferPoints(radius, bounds):
    def func_lws(pt):
        pt = ee.Feature(pt)
        return pt.buffer(radius).bounds() if bounds else pt.buffer(radius)
    return func_lws

points_buffered = awsPoints.map(bufferPoints(1000, True))
print(points_buffered.getInfo())


# Create a function computes zonal statistics of an ImageCollection over a FeatureCollection, including all
# necessary meta data. 
def zonalStats(ic, fc, params=None):
    'Compute zonal statistics of an ImageCollection over a FeatureCollection.'

    ## Initialize necessary parameters
    _params = {
        'reducer': ee.Reducer.mean(),
        'scale': None,
        'crs': None,
        'bands': None,
        'bandsRename': None,
        'imgProps': None,
        'imgPropsRename': None,
        'datetimeName': 'datetime',
        'datetimeFormat': 'YYYY-MM-dd HH:mm:ss'
    }

    ## Replace default params with (manually) provided params.
    if params:
        for k, v in params.items():
            _params[k] = v if v is not None else _params[k]

    ## Derive defaults from a representative image (first in collection)
    img_rep = ic.first()
    non_system_props = ee.Feature(None).copyProperties(img_rep).propertyNames()
    if _params['bands'] is None:
        _params['bands'] = img_rep.bandNames()
    if _params['bandsRename'] is None:
        _params['bandsRename'] = _params['bands']
    if _params['imgProps'] is None:
        _params['imgProps'] = non_system_props
    if _params['imgPropsRename'] is None:
        _params['imgPropsRename'] = _params['imgProps']

    ## Map over the ImageCollection
    def per_image(img):
        img = (ee.Image(img)
               .select(_params['bands'], _params['bandsRename'])
               .set(_params['datetimeName'],
                    img.date().format(_params['datetimeFormat']))
               .set('timestamp', img.get('system:time_start')))

        props_from = ee.List(_params['imgProps']) \
                       .cat(ee.List([_params['datetimeName'], 'timestamp']))
        props_to = ee.List(_params['imgPropsRename']) \
                       .cat(ee.List([_params['datetimeName'], 'timestamp']))
        img_props = img.toDictionary(props_from).rename(props_from, props_to)

        fc_sub = fc.filterBounds(img.geometry())

        def add_props(f):
            return ee.Feature(f).set(img_props).set('sampleID', f.get('name'))

        return img.reduceRegions(
            collection=fc_sub,
            reducer=_params['reducer'],
            scale=_params['scale'],
            crs=_params['crs']
        ).map(add_props)

    results = ic.map(per_image).flatten() \
                .filter(ee.Filter.notNull(_params['bandsRename'])) ### TRY WITHOUT THIS

    return results


##########
##########
##########

# Call zonalStats on all datasets separately. Stack threw errors (not all bands were found in every image) and loops are not recommended in Earth Engine.
results_mod_day = zonalStats(MOD11A1Daytime, points_buffered, {
        "reducer": ee.Reducer.mean(),
        "scale": 1000,
        "imgProps": ['system:index', 'system:time_start'],
        "datetimeName": 'date',
        "datetimeFormat": 'YYYY-MM-dd'
})

results_mod_night = zonalStats(MOD11A1Nighttime, points_buffered, {
        "reducer": ee.Reducer.mean(),
        "scale": 1000,
        "imgProps": ['system:index', 'system:time_start'],
        "datetimeName": 'date',
        "datetimeFormat": 'YYYY-MM-dd'
})

results_myd_day = zonalStats(MYD11A1Daytime, points_buffered, {
        "reducer": ee.Reducer.mean(),
        "scale": 1000,
        "imgProps": ['system:index', 'system:time_start'],
        "datetimeName": 'date',
        "datetimeFormat": 'YYYY-MM-dd'
})

results_myd_night = zonalStats(MYD11A1Nighttime, points_buffered, {
        "reducer": ee.Reducer.mean(),
        "scale": 1000,
        "imgProps": ['system:index', 'system:time_start'],
        "datetimeName": 'date',
        "datetimeFormat": 'YYYY-MM-dd'
})

lstData = results_mod_day.merge(results_mod_night).merge(results_myd_day).merge(results_myd_night)

### Output is again only returning MOD_Day band - just as in the loop before. 

print(ee.Feature(results_mod_day.first()).toDictionary().getInfo())
# print(ee.Feature(results_mod_night.first()).toDictionary().getInfo())     ERROR: "Element.toDictionary: Parameter 'element' is required and may not be null."
print(ee.Feature(results_myd_day.first()).toDictionary().getInfo())
# print(ee.Feature(results_myd_night.first()).toDictionary().getInfo())     ERROR: "Element.toDictionary: Parameter 'element' is required and may not be null."

{'type': 'FeatureCollection', 'columns': {'id': 'String', 'system:index': 'String'}, 'features': [{'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-51.39243991114472, 64.11348689092756], [-51.35147407304052, 64.11348689092756], [-51.35147407304052, 64.1314811657562], [-51.39243991114472, 64.1314811657562], [-51.39243991114472, 64.11348689092756]]]}, 'id': '0', 'properties': {'id': 'Kobbefjord_M500'}}, {'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-53.504618925038145, 69.26400721531922], [-53.45409990743221, 69.26400721531922], [-53.45409990743221, 69.28200149046323], [-53.504618925038145, 69.28200149046323], [-53.504618925038145, 69.26400721531922]]]}, 'id': '1', 'properties': {'id': 'Disko_T1'}}, {'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-53.458056385944, 69.28009760838441], [-53.407499846654176, 69.28009760838441], [-53.407499846654176, 69.29809188352962],

In [ ]:
# ATTEMPT 2: Zonal statistics function without defaults

# Create a buffer function 
def bufferPoints(radius, bounds):
    def func_lws(pt):
        pt = ee.Feature(pt)
        return pt.buffer(radius).bounds() if bounds else pt.buffer(radius)
    return func_lws

points_buffered = awsPoints.map(bufferPoints(1000, True))
print(points_buffered.getInfo())


# Compute zonal statistics of an ImageCollection over a FeatureCollection, without setting defaults or changing band names.
def zonalStats(ic, fc, params=None):
    'Compute zonal statistics of an ImageCollection over a FeatureCollection.'

    reducer = params.get('reducer') if params and 'reducer' in params else ee.Reducer.mean()
    scale = params.get('scale') if params and 'scale' in params else None
    crs = params.get('crs') if params and 'crs' in params else None

    def per_image(img):
        img = ee.Image(img)
        fc_sub = fc.filterBounds(img.geometry())
        return img.reduceRegions(
            collection=fc_sub,
            reducer=reducer,
            scale=scale,
            crs=crs
        )

    results = ic.map(per_image).flatten()
    return results

##########
##########
##########

# Call zonalStats on all datasets separately. Stack threw errors (not all bands were found in every image) and loops are not recommended in Earth Engine.
results_mod_day = zonalStats(MOD11A1Daytime, points_buffered, {
        "reducer": ee.Reducer.mean(),
        "scale": 1000,
        "imgProps": ['system:index', 'system:time_start'],
        "datetimeName": 'date',
        "datetimeFormat": 'YYYY-MM-dd'
})

results_mod_night = zonalStats(MOD11A1Nighttime, points_buffered, {
        "reducer": ee.Reducer.mean(),
        "scale": 1000,
        "imgProps": ['system:index', 'system:time_start'],
        "datetimeName": 'date',
        "datetimeFormat": 'YYYY-MM-dd'
})

results_myd_day = zonalStats(MYD11A1Daytime, points_buffered, {
        "reducer": ee.Reducer.mean(),
        "scale": 1000,
        "imgProps": ['system:index', 'system:time_start'],
        "datetimeName": 'date',
        "datetimeFormat": 'YYYY-MM-dd'
})

results_myd_night = zonalStats(MYD11A1Nighttime, points_buffered, {
        "reducer": ee.Reducer.mean(),
        "scale": 1000,
        "imgProps": ['system:index', 'system:time_start'],
        "datetimeName": 'date',
        "datetimeFormat": 'YYYY-MM-dd'
})

lstData = results_mod_day.merge(results_mod_night).merge(results_myd_day).merge(results_myd_night)
print(lstData.first().getInfo())


### Output is again only returning MOD_Day band - just as in the loop before. 

In [42]:
# ATTEMPT 3: Simpler approach using the reduceRegions method directly

# Create a buffer function 
def bufferPoints(radius, bounds):
    def func_lws(pt):
        pt = ee.Feature(pt)
        return pt.buffer(radius).bounds() if bounds else pt.buffer(radius)
    return func_lws

points_buffered = awsPoints.map(bufferPoints(1000, True))
print(points_buffered.getInfo())


mod_day = MOD11A1Daytime.select('MOD_LST_Day').map(
	lambda img: img.reduceRegions(collection=points_buffered, reducer=ee.Reducer.mean(), scale=1000)
).flatten()

# Inspect one feature
print(ee.Feature(mod_day.first()).toDictionary().getInfo())


mod_night = MOD11A1Nighttime.select('MOD_LST_Night').map(
	lambda img: img.reduceRegions(collection=points_buffered, reducer=ee.Reducer.mean(), scale=1000)
).flatten()

myd_day = MYD11A1Daytime.select('MYD_LST_Day').map(
	lambda img: img.reduceRegions(collection=points_buffered, reducer=ee.Reducer.mean(), scale=1000)
).flatten()

myd_night = MYD11A1Nighttime.select('MYD_LST_Night').map(
	lambda img: img.reduceRegions(collection=points_buffered, reducer=ee.Reducer.mean(), scale=1000)
).flatten()

lstData_ny = mod_day.merge(mod_night).merge(myd_day).merge(myd_night)


print(ee.Feature(mod_day.first()).toDictionary().getInfo())
print(ee.Feature(mod_night.first()).toDictionary().getInfo())
print(ee.Feature(myd_day.first()).toDictionary().getInfo())
print(ee.Feature(myd_night.first()).toDictionary().getInfo())

# This one does not extract cell values or zonal stats. Just returns systemtime, point id and geometry.

{'type': 'FeatureCollection', 'columns': {'id': 'String', 'system:index': 'String'}, 'features': [{'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-51.39243991114472, 64.11348689092756], [-51.35147407304052, 64.11348689092756], [-51.35147407304052, 64.1314811657562], [-51.39243991114472, 64.1314811657562], [-51.39243991114472, 64.11348689092756]]]}, 'id': '0', 'properties': {'id': 'Kobbefjord_M500'}}, {'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-53.504618925038145, 69.26400721531922], [-53.45409990743221, 69.26400721531922], [-53.45409990743221, 69.28200149046323], [-53.504618925038145, 69.28200149046323], [-53.504618925038145, 69.26400721531922]]]}, 'id': '1', 'properties': {'id': 'Disko_T1'}}, {'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-53.458056385944, 69.28009760838441], [-53.407499846654176, 69.28009760838441], [-53.407499846654176, 69.29809188352962],

In [ ]:
# ATTEMPT 4: Use geemap function to extract values to points

import os
work_dir = "C:/Users/simon/OneDrive - University of Copenhagen/Documents/Arbeitsmappe/GEMLST/GEMLST/Output"
out_csv = os.path.join(work_dir, "GEM_AWS_MODIS_LST_test02.csv")
lst_mod_day_2 = MOD11A1Daytime.select('MOD_LST_Day').map(
	lambda img: img.reduceRegions(collection=points_buffered, reducer=ee.Reducer.mean(), scale=1000)
).flatten()

print(ee.Feature(lst_mod_day_2.first()).toDictionary().getInfo())

# This one does not extract cell values or zonal stats. Just returns systemtime, point id and geometry. Would presumably only extract pixel values instead of zonal stats.

### Export

In [43]:
# # Export to Drive (CSV)
# task = ee.batch.Export.table.toDrive(
#     collection=results_mod_night,
#     description='GEM_AWS_MODIS_LST_test04',
#     fileFormat='CSV',
#     folder='gee'
# )
# task.start()

ee.batch.Task.list()


[<Task ZXXY7DPLKHLHBV7PKKU7L43C EXPORT_FEATURES: GEM_AWS_MODIS_LST_test04 (COMPLETED)>,
 <Task NFR4CPVOGAPYP2VF5JHMBJQF EXPORT_FEATURES: GEM_AWS_MODIS_LST_test02 (COMPLETED)>,
 <Task UO5T3O26JIFHU2DFOEY2QVYM EXPORT_FEATURES: GEM_AWS_MODIS_LST_test02 (COMPLETED)>,
 <Task E4XNK3NMSFUM4TEBMZBJQUC6 EXPORT_FEATURES: GEM_AWS_MODIS_LST_test01 (COMPLETED)>,
 <Task KSEPEY4GFYC7MXCJIW3UAKVE EXPORT_FEATURES: GEM_AWS_MODIS_LST_test01 (FAILED)>]

In [35]:
image_test_night = MOD11A1Nighttime.select('LST_Night_1km').mean()

In [38]:
map2 = geemap.Map()

maskedDay = maskQualityDaytime(test_image2)
maskedNight = maskQualityNighttime(test_image2)

vis_params_d = {'bands': ['LST_Day_1km_C'], 'min': -60, 'max': 30, 'palette': 'coolwarm'}
vis_params_n = {'bands': ['LST_Night_1km_C'], 'min': -60, 'max': 30, 'palette': 'coolwarm'}


# map2.addLayer(masked, {"bands": ['LST_Day_1km'], "min": 13000, "max": 16500, "palette": ['blue', 'green', 'red']}, 'Masked LST Day 1km')
# map2.addLayer(maskedDay, vis_params_d, 'masked LST Day 1km')
# map2.addLayer(test_image2, vis_params_d, 'LST Day 1km')
# map2.addLayer(maskedNight, vis_params_n, 'masked LST Night 1km')
# map2.addLayer(test_image2, vis_params_n, 'LST Night 1km')
map2.addLayer(image_test_night, {"bands": ['LST_Night_1km'], "min": 0, "max": 16500, "palette": ['blue', 'green', 'red']}, 'Mean LST Night 1km')
map2.addLayer(awsPoints, {"color": 'red'}, 'AWS Points')
map2.centerObject(awsPoints, 4)
map2


Map(center=[71.18782603850234, -43.309990026341694], controls=(WidgetControl(options=['position', 'transparent…